# Exercise 58 — Filtering and sorting

## Concept

### Filtering — two equivalent styles

```python
# Boolean mask (the workhorse)
df[(df["amount"] > 100) & (df["paid"])]

# .query() — string expression, can be more readable for simple filters
df.query("amount > 100 and paid")
df.query("region in ['North', 'East']")

# Reference a Python variable inside .query with @
threshold = 100
df.query("amount > @threshold")
```

Both produce the same result. Mask is more flexible (you can use any vectorized expression); `.query()` is shorter for simple cases. Pick by readability — there's no performance difference worth caring about.

### Sorting

```python
df.sort_values("amount")                                # ascending by default
df.sort_values("amount", ascending=False)               # descending
df.sort_values(["region", "amount"],                    # multi-column
               ascending=[True, False])                 # region asc, amount desc

df.sort_values("amount", na_position="last")            # nulls last (default), or 'first'
```

All non-mutating — returns a new DataFrame. Use `df.sort_values(...).reset_index(drop=True)` if you want a fresh `0..n-1` index after sorting (otherwise the original row labels come along).

## Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "customer": ["Alice", "Bob", "Alice", "Carol", "Bob", "Diana", "Alice", "Carol", "Bob", "Eve"],
    "region":   ["N", "S", "N", "S", "N", "E", "S", "N", "S", "E"],
    "amount":   [120.0, 45.0, 300.0, 89.0, 210.0, 55.0, 430.0, 12.0, np.nan, 750.0],
    "paid":     [True, True, False, True, True, False, True, True, False, True],
})
print(df)


## Task 1 — Filter with `.query`

Return rows where `amount` is greater than `threshold` AND `paid` is `True`. Use `.query()` (with `@threshold` to inject the parameter).

```python
def paid_above(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    ...
```

Expected: `paid_above(df, 200)` returns order_ids `{5, 7, 10}` (210, 430, 750).

In [ ]:
import pandas as pd

def paid_above(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    pass  # TODO

result = paid_above(df, 200)
assert set(result["order_id"]) == {5, 7, 10}
print("ok")


## Task 2 — Multi-column sort

Return the DataFrame sorted by `region` ascending and then `amount` **descending**, with the original index reset (so `.index` is `0..n-1`).

```python
def ranked(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Nulls in `amount` should come **last** within each region.

In [ ]:
import pandas as pd

def ranked(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = ranked(df)
assert result.index.tolist() == list(range(len(df)))
# Within E: 750, 55
assert result.query("region == 'E'")["amount"].tolist() == [750.0, 55.0]
# Within N: 300, 210, 120, 12
assert result.query("region == 'N'")["amount"].tolist() == [300.0, 210.0, 120.0, 12.0]
# Within S: 430, 89, 45, NaN (last)
south = result.query("region == 'S'")["amount"].tolist()
assert south[:3] == [430.0, 89.0, 45.0]
assert pd.isna(south[3])
print("ok")


## Task 3 — Top-N per group (without groupby yet)

Return the **top-2 orders by amount overall** (just `nlargest`, no per-region grouping yet).

```python
def top_n_orders(df: pd.DataFrame, n: int) -> pd.DataFrame:
    ...
```

Hint: `df.nlargest(n, "amount")` is the shortcut — internally it's a sort + head, but the method exists because it's so common.

Expected: `top_n_orders(df, 2)` returns order_ids `[10, 7]` (750, 430).

In [ ]:
import pandas as pd

def top_n_orders(df: pd.DataFrame, n: int) -> pd.DataFrame:
    pass  # TODO

result = top_n_orders(df, 2)
assert result["order_id"].tolist() == [10, 7]
assert result["amount"].tolist() == [750.0, 430.0]
print("ok")


## Task 4 — Boolean mask combining multiple conditions

Return rows where:
- `region` is `"N"` or `"S"`, AND
- `amount` is not null, AND
- (`paid` is `True` OR `amount > 100`)

Use boolean masks with `&`, `|`, `~`, parenthesizing each comparison.

```python
def complex_filter(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected order_ids: `{1, 2, 3, 4, 5, 7, 8}` (everything in N/S with non-null amount that is either paid or >100 — row 9 fails on `amount` being NaN; the East rows fail the region check).

In [ ]:
import pandas as pd

def complex_filter(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = complex_filter(df)
assert set(result["order_id"]) == {1, 2, 3, 4, 5, 7, 8}
print("ok")
